# Task
Run AI text detection tests on texts from "fake_texts.json" and "real_texts.json" using the following Hugging Face models: "fakespot-ai/roberta-base-ai-text-detection-v1", "desklib/ai-text-detector-v1.01", "aartik001/ai_text_detector", "Juner/AI-generated-text-detection", and "desklib/ai-text-detector-academic-v1.01". Load each model, run the tests, and then delete the model from GPU memory before proceeding to the next model.

## Setup and data loading

### Subtask:
Install necessary libraries and load the text data from the two JSON files into a suitable format.


**Reasoning**:
The first step is to install the necessary libraries.



In [ ]:
%pip install datasets transformers

**Reasoning**:
Load the data from the JSON files into datasets and concatenate them, then add a label column.



# Task
Run AI text detection tests on texts from JSON files in "/content/reports" using the Hugging Face models "fakespot-ai/roberta-base-ai-text-detection-v1", "desklib/ai-text-detector-v1.01", "aartik001/ai_text_detector", "Juner/AI-generated-text-detection", and "desklib/ai-text-detector-academic-v1.01". Load each model, run predictions on the 'container' and 'encoded_message' fields of the JSON data, evaluate the accuracy of the AI predictions, and clear the GPU memory before processing the next model. Finally, compare the accuracy of the models.

## Setup and data loading

### Subtask:
Install necessary libraries and load the text data from the two JSON files into a suitable format.


**Reasoning**:
Load the JSON data from the specified directory and create a Hugging Face Dataset.



In [1]:
import os
import json
from datasets import Dataset

json_dir = "/content/reports"
json_files = [os.path.join(json_dir, f) for f in os.listdir(json_dir) if f.endswith(".json")]

# Load data from each JSON file
before_synsteg, after_synsteg = [], []
for json_file in json_files:
    with open(json_file) as f:
      report = json.load(f)

    before_synsteg.append(report["container"])
    after_synsteg.append(report["encoded_message"])

# Combine the texts and create labels (both are AI-generated)
texts = before_synsteg + after_synsteg
metadata = ["before_synsteg" for text in before_synsteg] + ["after_synsteg" for text in after_synsteg]
labels = ["AI"] * len(texts)

# Create a Hugging Face Dataset
data = Dataset.from_dict({"text": texts, "label": labels, "meta": metadata})
display(data)

Dataset({
    features: ['text', 'label', 'meta'],
    num_rows: 1200
})

**Reasoning**:
Create a function to load a Hugging Face model and its tokenizer to the GPU.

In [2]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def load_model_and_tokenizer(model_name):
    """Loads a Hugging Face model and tokenizer to the GPU."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    return tokenizer, model, device

**Reasoning**:
Create a function to perform AI text detection predictions.

In [10]:
from transformers import pipeline

def get_predictions(tokenizer, model, device, dataset, token_size):
    """Performs predictions on the dataset using the provided model and tokenizer."""
    # Explicitly set max_length for truncation to avoid tensor size mismatch
    pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device, truncation=True, max_length=token_size)
    # Using batched processing for efficiency
    predictions = pipe(list(dataset["text"]), batch_size=16)
    return predictions

**Reasoning**:
Create a function to evaluate the accuracy of the model's predictions.

In [4]:
from sklearn.metrics import accuracy_score

def evaluate_accuracy(predictions, dataset):
    """Evaluates the accuracy of the predictions against the dataset labels."""
    # Extract predicted labels (assuming the model outputs 'AI' or 'Human')
    predicted_labels = [p['label'] for p in predictions]

    # The true labels are all 'AI' for this dataset as per the problem description
    true_labels = dataset["label"]

    # Calculate accuracy
    accuracy = accuracy_score(true_labels, predicted_labels)
    return accuracy

**Reasoning**:
Implement a mechanism to clear the GPU memory after using each model.

In [5]:
import gc

def clear_gpu_memory(model, tokenizer):
    """Clears the GPU memory occupied by the model and tokenizer."""
    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()

**Reasoning**:
Iterate through the models, perform predictions, evaluate accuracy, and clear GPU memory for each.

In [11]:
model_names = {
    "fakespot-ai/roberta-base-ai-text-detection-v1": 512,
    "Juner/AI-generated-text-detection": 512,
}

accuracy_results = {}

for model_name, token_length in model_names.items():
    print(f"Processing model: {model_name}")
    try:
        tokenizer, model, device = load_model_and_tokenizer(model_name)

        # Filter data by meta for separate accuracy calculation
        data_before = data.filter(lambda x: x['meta'] == 'before_synsteg')
        data_after = data.filter(lambda x: x['meta'] == 'after_synsteg')

        # Get predictions and evaluate accuracy for 'before_synsteg'
        predictions_before = get_predictions(tokenizer, model, device, data_before, token_length)
        accuracy_before = evaluate_accuracy(predictions_before, data_before)

        # Get predictions and evaluate accuracy for 'after_synsteg'
        predictions_after = get_predictions(tokenizer, model, device, data_after, token_length)
        accuracy_after = evaluate_accuracy(predictions_after, data_after)

        accuracy_results[model_name] = {
            "before_synsteg": accuracy_before,
            "after_synsteg": accuracy_after
        }
        print(f"Accuracy for {model_name}:")
        print(f"  Before Synsteg: {accuracy_before:.4f}")
        print(f"  After Synsteg: {accuracy_after:.4f}\n")

        clear_gpu_memory(model, tokenizer)
    except Exception as e:
        print(f"Skipping model {model_name} due to error: {e}\n")
        # Ensure memory is cleared even if loading failed
        if 'model' in locals() and model is not None:
            del model
        if 'tokenizer' in locals() and tokenizer is not None:
            del tokenizer
        torch.cuda.empty_cache()
        gc.collect()

Processing model: fakespot-ai/roberta-base-ai-text-detection-v1


Filter:   0%|          | 0/1200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1200 [00:00<?, ? examples/s]

Device set to use cuda
Device set to use cuda


Accuracy for fakespot-ai/roberta-base-ai-text-detection-v1:
  Before Synsteg: 0.9983
  After Synsteg: 0.6867

Processing model: Juner/AI-generated-text-detection


Filter:   0%|          | 0/1200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1200 [00:00<?, ? examples/s]

Device set to use cuda
Device set to use cuda


Accuracy for Juner/AI-generated-text-detection:
  Before Synsteg: 0.0000
  After Synsteg: 0.0000

